# 12 · Préparation des données — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch12_preparation/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 12.Q1 à 12.Q11, 12.R1 à 12.R3, 12.1 à 12.5, 12.7 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★★ | |
| A | 12.11 à 12.15 | diagnostiquer Penguins brut, un CSV à la française, où tombent les données de test après un MinMaxScaler, dates, jointures et tableaux croisés avec pandas, une base SQL (*prochaine session de génération*) | | | |
| B | 12.16 à 12.21 | StandardScaler, MinMaxScaler et SimpleImputer dans mylearn.preprocessing, scalers face aux points aberrants, bugs de préparation, transformer la cible (*prochaine session de génération*) | | | |
| C | 12.22 à 12.28 | projeter un nuage, variance expliquée, MNIST en composantes, sélection de features, OrdinalEncoder, OneHotEncoder et PCA dans mylearn.preprocessing (*prochaine session de génération*) | | | |
| D | 12.29 à 12.33 | échelle et descente de gradient, PCA, t-SNE et UMAP, fuites par le prétraitement, une préparation testée, défi Penguins brut (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Diagnostiquer et nettoyer un dataset brut avec pandas, et l'assembler avec des jointures et SQL.
- Écrire et vérifier contre scikit-learn les transformateurs de `mylearn.preprocessing` : `StandardScaler`, `MinMaxScaler`, `SimpleImputer`, `OrdinalEncoder`, `OneHotEncoder` et `PCA`.
- Choisir une mise à l'échelle et un encodage, et appliquer la règle d'or : `fit` sur l'entraînement, `transform` partout.
- Réduire et visualiser des données en grande dimension (PCA, t-SNE, UMAP), et mesurer les fuites dues au prétraitement.

**Rappel express.** Min-max : $x' = a + (b - a)\,\frac{x - \min}{\max - \min}$ ; standardisation : $z = \frac{x - \mu}{\sigma}$ (ddof = 0), inverse $x = z\,\sigma + \mu$ ; covariance : $\boldsymbol{\Sigma} = \frac{1}{n - 1}\mathbf{X}_c^\top\mathbf{X}_c$ ; variance des projections sur $\mathbf{u}$ unitaire : $\mathbf{u}^\top\boldsymbol{\Sigma}\,\mathbf{u}$ ; PCA : $\mathbf{Z} = (\mathbf{X} - \boldsymbol{\mu})\,\mathbf{W}_k^\top$, $\hat{\mathbf{X}} = \mathbf{Z}\,\mathbf{W}_k + \boldsymbol{\mu}$, part de variance expliquée $\lambda_j / \sum_l \lambda_l$. En Python : `sklearn.preprocessing` (`StandardScaler`, `MinMaxScaler`, `RobustScaler`, `OrdinalEncoder`, `OneHotEncoder`), `sklearn.impute.SimpleImputer`, `sklearn.decomposition.PCA`, `sklearn.pipeline.make_pipeline`, `np.linalg.svd`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 12.1 à 12.5 et 12.7)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix : les lettres collées (`"BF"`) ; une suite de codes : les lettres dans l'ordre (`"XYZ"`) ; plusieurs nombres : une liste (`[4, 7]`) ; une matrice : une liste de lignes (`[[1, 1], [1, 1]]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 12.6 et 12.8, la réflexion (🗣️ 12.9, ⚖️ 12.10) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np


def minmax(x, lo, hi, a=0.0, b=1.0):
    """Min-max scaling of x with the training minimum lo and maximum hi, towards [a, b]."""
    return a + (b - a) * (np.asarray(x, dtype=float) - lo) / (hi - lo)


# Q6: training temperatures from -5 to 25 degrees
Q6_30 = float(minmax(30, -5, 25))

# R3: f(x, y) = x^2 + 100 y^2
GRAD_R3 = [2 * 1, 200 * 1]

# 12.2: five training temperatures
T_122 = np.array([14.0, 17.0, 21.0, 23.0, 25.0])
MU_122, SD0_122, SD1_122 = T_122.mean(), T_122.std(ddof=0), T_122.std(ddof=1)

# 12.3: three feature ranges and one example
LO_123, HI_123 = np.array([10.0, 0.0, -20.0]), np.array([30.0, 5.0, 20.0])
X_123 = np.array([20.0, 4.0, 0.0])
GLO_123, GHI_123 = LO_123.min(), HI_123.max()

# 12.4: temperatures from -18 to 12, cars from 120 to 870
T_LO, T_HI, C_LO, C_HI = -18.0, 12.0, 120.0, 870.0

# 12.5: a 3 x 3 training table
X_125 = np.array([[1.0, 4.0, 10.0], [3.0, 8.0, 20.0], [5.0, 6.0, 30.0]])
NEW_125 = np.array([2.0, 2.0, 40.0])
COL_LO, COL_HI = X_125.min(axis=0), X_125.max(axis=0)


def row_minmax(row):
    return (row - row.min()) / (row.max() - row.min())


# 12.7: five points, their covariance and the principal direction (2, 1) / sqrt(5)
P_127 = np.array([[1.0, 1.0], [3.0, 3.0], [4.0, 3.0], [5.0, 5.0], [7.0, 3.0]])
MU_127 = P_127.mean(axis=0)
XC_127 = P_127 - MU_127
COV_127 = XC_127.T @ XC_127 / (len(P_127) - 1)
U_127 = np.array([2.0, 1.0]) / math.sqrt(5)
V_127 = np.array([-1.0, 2.0]) / math.sqrt(5)
LAMBDA_127 = float((COV_127 @ np.array([2.0, 1.0]))[0] / 2.0)
TOTAL_127 = float(np.trace(COV_127))
T_127 = XC_127 @ U_127
D_127 = P_127[3]
T_D = float((D_127 - MU_127) @ U_127)
REC_D = MU_127 + T_D * U_127

**12.Q1 — La règle d'or de la préparation**

In [3]:
wb.record("12.Q1a", "B", mistakes={"les données du test entrent alors dans le calcul de la moyenne : c'est une fuite (fiche §12.2 et §12.10)": "A",
          "le modèle a appris sur des données transformées avec certains paramètres : lesquels faut-il réutiliser ?": "C",
          "avec deux jeux de paramètres, un même nombre n'a plus le même sens à l'entraînement et au test": "D"})
wb.record("12.Q1b", "B", mistakes={"réapprendre les paramètres sur une seule mesure change le sens des nombres que reçoit le modèle": "A",
          "le modèle a appris sur des données transformées : il ne connaît pas les unités brutes": "C",
          "réapprendre les paramètres sur d'autres données, même nombreuses, change le sens des nombres que reçoit le modèle": "D"})
wb.record("12.Q1c", False, mistakes={"couper les silences fait partie de la préparation : est-ce encore fait en service ?": True})
wb.record("12.Q1d", True, mistakes={"toute étape de préparation, une colonne retirée comprise, fait partie du modèle (fiche §12.2)": False})

WB_ANSWER {"hash": "f65c12c1ce14e11df06adbbe67b2498ad5aa426cd49dd7dcec024908679b4f77", "id": "12.Q1a", "kind": "str", "mistakes": {"0fd6dd4426810be347715aaf61f437c8e72e9b3ba57d89b211136136a26194f8": "les données du test entrent alors dans le calcul de la moyenne : c'est une fuite (fiche §12.2 et §12.10)", "2b86181f85fd7071fb9107c2906de71ddefdaed13054d736bfd26affda62166a": "le modèle a appris sur des données transformées avec certains paramètres : lesquels faut-il réutiliser ?", "6f630aacf48584188373c17b829a6474e0d6a49dcbbefec476f3fc5eb5215299": "avec deux jeux de paramètres, un même nombre n'a plus le même sens à l'entraînement et au test"}}
📝 Ex 12.Q1a : réponse enregistrée (str).
WB_ANSWER {"hash": "592d3620a6ecb3476bfafe4bd76813dc6e3cd4f51883f47f043f8813095889be", "id": "12.Q1b", "kind": "str", "mistakes": {"0b6dea2e791141f57af274cf96cabd4c67548e8a0e0156193ca977b1ea4d9ea6": "le modèle a appris sur des données transformées : il ne connaît pas les unités brutes", "a80e4af34f30ac6f4db3

**12.Q2 — Numérique, ordinale ou nominale ?**

In [4]:
wb.record("12.Q2a", "QONNOQ", mistakes={"élément 4 (le code postal) : des chiffres ne font pas un nombre ; une moyenne de codes postaux aurait-elle un sens ?": "QONQOQ",
          "élément 4 (le code postal) : un ordre entre deux codes postaux aurait-il un sens pour le problème ?": "QONOOQ",
          "élément 2 (les avis) : ces trois réponses ont-elles un ordre qui a un sens ?": "QNNNOQ",
          "élément 5 (les tailles) : ces catégories ont-elles un ordre naturel ?": "QONNNQ",
          "élément 6 (le nombre d'enfants) : c'est un nombre, qui se compare et se calcule": "QONNOO",
          "élément 3 (l'île) : quel ordre aurait un sens entre des îles ?": "QOONOQ"})
wb.record("12.Q2b", "B", mistakes={"c'est l'ordre naturel, mais l'encodeur ne le connaît pas : selon quel critère trie-t-il sans indication ?": "A",
          "l'encodeur range bien les catégories, mais pas par taille : selon quel critère, sans indication ?": "C",
          "l'encodeur trie les catégories qu'il a vues : selon quel critère ?": "D"})
wb.record("12.Q2c", False, mistakes={"un ordre inventé permet de trier, pas de donner un sens à « entre » (encadré ⚠️ du §12.3)": True})

WB_ANSWER {"hash": "2b64009dac6ccb62e3bf290b9a96d11fd6e5d98c960e91ed60c9608cb3e7aa50", "id": "12.Q2a", "kind": "str", "mistakes": {"093fef8806dba9c843174fa88df049d0ba46967f1dfee1489ce04815580bd5cd": "élément 4 (le code postal) : un ordre entre deux codes postaux aurait-il un sens pour le problème ?", "0b72977345ec0df2f7a8b99bf14faef1ca0b4ab01f58a98e208d5dcb88f0ae15": "élément 2 (les avis) : ces trois réponses ont-elles un ordre qui a un sens ?", "38003a610798ee9778d7ad1cf9975295a53881913ab5d5195adbd83e89677a8e": "élément 3 (l'île) : quel ordre aurait un sens entre des îles ?", "612a57e286b10225cf496844029458a9c356502a711f1b82ce773a8520126aff": "élément 6 (le nombre d'enfants) : c'est un nombre, qui se compare et se calcule", "7b80e5d40d08c7faa0bba42bd5065a6c00da7239a7c668e255bfe240447a63e5": "élément 5 (les tailles) : ces catégories ont-elles un ordre naturel ?", "7d67c12ed490ee42e17b2f84c49eba2aa5ac05bff3b29e4261455dea24ff8f9a": "élément 4 (le code postal) : des chiffres ne font pas u

**12.Q3 — Pourquoi un one-hot plutôt qu'un entier ?**

In [5]:
wb.record("12.Q3a", "A", mistakes={"de train (1) à tram (2), de combien le code augmente-t-il ? Et le terme w × code ?": "B",
          "ce n'est le cas que si w vaut 0": "C",
          "avec une seule colonne et un seul poids, chaque catégorie peut-elle avoir son effet propre ?": "D"})
wb.record("12.Q3b", 3, mistakes={"sans drop, chaque catégorie a sa propre colonne": 2,
          "un one-hot ne garde pas une seule colonne : relis sa définition": 1})
wb.record("12.Q3c", 2, mistakes={"drop=\"first\" retire une colonne : compte celles qui restent": 3,
          "drop=\"first\" ne retire qu'une seule colonne": 1})
wb.record("12.Q3d", [0, 0, 1], mistakes={"c'est le vecteur de la première catégorie dans l'ordre alphabétique ; où se range tram ?": [1, 0, 0],
          "train et tram commencent par les mêmes lettres : compare-les lettre à lettre": [0, 1, 0]})
wb.record("12.Q3e", True, mistakes={"compare les écarts entre les codes de bus et de tram, puis de bus et de train": False})

WB_ANSWER {"hash": "d57f853135f3a843d52d887b3c9b0fae92d0ec14c699122fd5f92e29d97da5ad", "id": "12.Q3a", "kind": "str", "mistakes": {"73a6d5cf74a028ab45b6d00597983478234fec99b8bd5052e72701f733023e58": "de train (1) à tram (2), de combien le code augmente-t-il ? Et le terme w × code ?", "a24bff3e109b0e364c753d9d313e29ef9da81989cc7bac0a10fec1ba3ee150ad": "ce n'est le cas que si w vaut 0", "c7c4b9090d7a01b50fe74d8144dce4be336821303f930a598bc70c8feb96a4dd": "avec une seule colonne et un seul poids, chaque catégorie peut-elle avoir son effet propre ?"}}
📝 Ex 12.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "f68ed52838c048581681094465d7ef3d2c8379138cec66c271d710dce2b7a4ed", "id": "12.Q3b", "kind": "int", "magnitude": 0, "mistakes": {"5b2164aacc821b60b94b96a2627950d9650b73d8f8e6ec433585cac9dc653800": "sans drop, chaque catégorie a sa propre colonne", "863720a71f21509f43107d0aa6a46332b702fcb12b98543d3b6fd62be27a0dc7": "un one-hot ne garde pas une seule colonne : relis sa définition"}, "sig

**12.Q4 — Doublons, NaN et points aberrants**

In [6]:
wb.record("12.Q4a", "B", mistakes={"38 kg pour une espèce qui pèse de 3 à 5 kg : est-ce plausible ?": "A",
          "une valeur manquante est une case vide ou un NaN ; ici, une valeur est présente": "C",
          "un doublon est une ligne répétée ; ici, les quatre masses sont différentes": "D"})
wb.record("12.Q4b", True, mistakes={"une ligne répétée entre deux fois dans chaque somme : moyenne, loss, vote": False})
wb.record("12.Q4c", False, mistakes={"pour pandas, « ? » est un texte comme un autre : quelle option de read_csv lui dit d'y voir une valeur manquante ?": True})
wb.record("12.Q4d", "C", mistakes={"Python lit le point comme séparateur décimal ; et la virgule ?": "A",
          "float() ne saute pas les caractères qu'il ne comprend pas : relis les formats dans la fiche §12.4": "B",
          "float() ne renvoie jamais 0 pour un texte illisible : relis les formats dans la fiche §12.4": "D"})
wb.record("12.Q4e", False, mistakes={"NaN n'est égal à rien, pas même à lui-même : que vaut np.nan == np.nan ?": True})

WB_ANSWER {"hash": "d3052a4997007c7d716a0bb5c5e0d4f2bb52df55ebf8acb133c90dc8b7f1a451", "id": "12.Q4a", "kind": "str", "mistakes": {"032b07ec451934ac5eddf021155f0492ad92eaf6bc438b9c0b3b510be6da1225": "38 kg pour une espèce qui pèse de 3 à 5 kg : est-ce plausible ?", "524cf12fad88bbcfe7449dc60d2459eada7c3ac5b7b11d1ecc93fb16dc802b9a": "une valeur manquante est une case vide ou un NaN ; ici, une valeur est présente", "592655d82dcde1b83e10693a9897dec7070eb16f7fa58837a48a43aa2b15f982": "un doublon est une ligne répétée ; ici, les quatre masses sont différentes"}}
📝 Ex 12.Q4a : réponse enregistrée (str).
WB_ANSWER {"hash": "3c3e02f157c3ed26b8693e8202971b55720674d72c34754c07e2cbf2cb97b108", "id": "12.Q4b", "kind": "bool", "mistakes": {"ab143b5c83d79c4a4f2e9f20d5d384e03e8c5edc23c38582193c716e0d13b21b": "une ligne répétée entre deux fois dans chaque somme : moyenne, loss, vote"}}
📝 Ex 12.Q4b : réponse enregistrée (bool).
WB_ANSWER {"hash": "a92712f5fe788763ad5b1dba40cd3d1655b096c6ffae0bae19dd07e

**12.Q5 — Normaliser ou standardiser ?**

In [7]:
wb.record("12.Q5a", "B", mistakes={"le min-max divise par l'étendue, ici fixée par la valeur 1 000 : où tombent alors les petites valeurs ?": "A",
          "le min-max n'envoie aucune valeur au milieu par principe : relis sa formule": "C",
          "le min-max vers [0, 1] ne donne aucune valeur négative sur l'entraînement": "D"})
wb.record("12.Q5b", False, mistakes={"un z-score n'est pas borné par 3 : pense à 1 000 valeurs dont une très éloignée des autres": True})
wb.record("12.Q5c", False, mistakes={"la standardisation déplace et dilate l'histogramme : change-t-elle sa forme ?": True})
wb.record("12.Q5d", True, mistakes={"une transformation affine déplace et dilate l'histogramme sans changer sa forme": False})

WB_ANSWER {"hash": "8039fcb4ff708988efdfaa57d9bf039302dec43b0d5426c86c5ff5006f2c6ca2", "id": "12.Q5a", "kind": "str", "mistakes": {"001ed442f8da6d30040bf0333bde49dcba7b75126d6e67337efc64e479b83f33": "le min-max divise par l'étendue, ici fixée par la valeur 1 000 : où tombent alors les petites valeurs ?", "406d28fab2cbc8a55244e26496c6eccfc5c93a1ffef4bb4f4631e01e3486f1ab": "le min-max vers [0, 1] ne donne aucune valeur négative sur l'entraînement", "751854b56124cb25e23e0470abe968b47cc35890763f47aab25dbc19feffb025": "le min-max n'envoie aucune valeur au milieu par principe : relis sa formule"}}
📝 Ex 12.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "2ae0f1ddd9537e588c4d8f05dcf82c754bce0a1273ff8dc09046f6087927a911", "id": "12.Q5b", "kind": "bool", "mistakes": {"983a803b2090594376dff39f4d6ace7cb9b72edfade03c18b563266072a4e205": "un z-score n'est pas borné par 3 : pense à 1 000 valeurs dont une très éloignée des autres"}}
📝 Ex 12.Q5b : réponse enregistrée (bool).
WB_ANSWER {"hash": "653

**12.Q6 — Données de test hors de [0, 1] : bug ou normal ?**

In [8]:
wb.record("12.Q6a", Q6_30, decimals=2, mistakes={"sans option, transform ne borne rien ; et le numérateur retire-t-il bien le minimum ?": 1.0,
          "le numérateur retire le minimum, pas le maximum": 0.17,
          "l'étendue vaut max − min, avec un minimum négatif : attention au signe": 1.75,
          "le numérateur retire le minimum, qui est négatif : attention au signe": 0.83,
          "si 1,2 est un arrondi, garde 2 décimales ; sinon, le minimum est-il retiré au numérateur et au dénominateur ?": 1.2,
          "le minimum vaut −5 : le retirer, c'est ajouter 5, au numérateur comme au dénominateur": 1.25,
          "le dénominateur est l'étendue max − min, pas le maximum": 1.4})
wb.record("12.Q6b", False, mistakes={"une valeur plus chaude que toutes celles de l'entraînement dépasse 1 : relis la fiche §12.5.3": True})
wb.record("12.Q6c", 1.0, decimals=2, mistakes={"clip=True ramène les valeurs aux bornes de feature_range": round(Q6_30, 2),
          "clip ramène une valeur trop grande à la borne la plus proche : laquelle ?": 0})
wb.record("12.Q6d", "B", mistakes={"le test n'a pas servi à calculer la moyenne : pourquoi aurait-il une moyenne exactement nulle ?": "A",
          "le scaler fixe l'écart-type de l'entraînement à 1, pas la moyenne du test": "C",
          "rien n'oblige le test à être plus bas que l'entraînement": "D"})

WB_ANSWER {"decimals": 2, "hash": "fc9d6cbc1cdc9a05e7ab4dd6a065922eca2b5f6586c00a9e532f16b389ecb289", "hash_coarse": "119c5853f012fbdf81e5e38a75c983a4c08568096608b456f271981955d3ccfe", "id": "12.Q6a", "kind": "float", "magnitude": 0, "mistakes": {"01b89341f831c3be7022f6c2f92f22bb162ec3e9ebeaa6c1ac2d9c46926f471b": "sans option, transform ne borne rien ; et le numérateur retire-t-il bien le minimum ?", "23145a930da44e47dc2f205d5c7883aecb13cb97d3a6ef33f53f356da50be7ae": "l'étendue vaut max − min, avec un minimum négatif : attention au signe", "2ceb1134964b1c64ba50ea9a1d077962a4d8d186e603f3f847e2390ab562411e": "le numérateur retire le minimum, pas le maximum", "366cb5aef621735e5d4591172db8c770def9ae8df70f49497b1cb2fc25454da9": "le minimum vaut −5 : le retirer, c'est ajouter 5, au numérateur comme au dénominateur", "96b12cdc945a1215deec1fd14d0d38c073a320a91eed1310fafb6d24a3f243cd": "le dénominateur est l'étendue max − min, pas le maximum", "b89d63d7c7955d75363cb42f5bb815308479ce9802351d6610

**12.Q7 — Univarié ou multivarié ?**

In [9]:
wb.record("12.Q7a", "UMMUMU", mistakes={"élément 1 (StandardScaler) : la moyenne et l'écart-type d'une colonne utilisent-ils les autres colonnes ?": "MMMUMU",
          "élément 3 (le min-max global) : le minimum et le maximum viennent de toutes les colonnes ensemble": "UMUUMU",
          "élément 5 (KNNImputer) : pour trouver les plus proches voisins, on compare les autres features de l'exemple": "UMMUUU",
          "élément 6 (le logarithme) : le logarithme d'une valeur dépend-il des autres colonnes ?": "UMMUMM",
          "élément 4 (la médiane) : la médiane d'une colonne se calcule-t-elle avec les autres colonnes ?": "UMMMMU",
          "élément 2 (la PCA) : une direction principale combine plusieurs features": "UUMUMU"})
wb.record("12.Q7b", False, mistakes={"univariée : chaque colonne est transformée avec ses seules valeurs": True})

WB_ANSWER {"hash": "18ad1dda8f07d50a0af060685bfe2246f4868d16755b3b8cade7dd9c21a16f97", "id": "12.Q7a", "kind": "str", "mistakes": {"3053dcf5e33f1b7108f8e4436542d74538782aa2b47c6c52826a2224c25ea323": "élément 1 (StandardScaler) : la moyenne et l'écart-type d'une colonne utilisent-ils les autres colonnes ?", "606b21259edc5c790ce42b80a010d9fb0f40ab3a95148f85a46a4d27d18cde78": "élément 3 (le min-max global) : le minimum et le maximum viennent de toutes les colonnes ensemble", "83e9c53df07a62f40893ee21362283f75d8af8bc14f238f449ed43b279481bbb": "élément 5 (KNNImputer) : pour trouver les plus proches voisins, on compare les autres features de l'exemple", "997239a7ab12185533714752a277b1863cdec8560df5c46f2b1296621eb55b69": "élément 6 (le logarithme) : le logarithme d'une valeur dépend-il des autres colonnes ?", "b15745f2d6d1731ee87b49e7a976903cc8ac8a3504006cd54e7497d6288a6cc9": "élément 4 (la médiane) : la médiane d'une colonne se calcule-t-elle avec les autres colonnes ?", "c8831bb8b3443bea089

**12.Q8 — Sélectionner ou réduire la dimension ?**

In [10]:
wb.record("12.Q8a", "SRSRS", mistakes={"élément 1 (la colonne constante) : les colonnes qui restent sont-elles modifiées ?": "RRSRS",
          "élément 4 (l'IMC) : garde-t-on une colonne d'origine, ou en fabrique-t-on une nouvelle ?": "SRSSS",
          "élément 3 (les 5 features les plus corrélées) : ces colonnes sont-elles modifiées ?": "SRRRS",
          "élément 5 (VarianceThreshold) : que fait-il des colonnes qu'il garde ?": "SRSRR",
          "élément 2 (la PCA) : ses composantes sont-elles des colonnes d'origine ?": "SSSRS"})
wb.record("12.Q8b", False, mistakes={"la cible du test a servi à choisir les features : relis 8.25 et la fiche §12.6": True})
wb.record("12.Q8c", False, mistakes={"une composante est une somme pondérée de toutes les features": True})

WB_ANSWER {"hash": "0157f216da501d798410156346aa48d0881aef5f4652ff144f8ded76d3d6d822", "id": "12.Q8a", "kind": "str", "mistakes": {"214905338c471d4ff564ce34cde83e07373ff8a289c8ea1d7a65864aa3782321": "élément 4 (l'IMC) : garde-t-on une colonne d'origine, ou en fabrique-t-on une nouvelle ?", "3135b5206d1fd25b05f944b5dd749d3983122ac6560e18c9bda88b92d611098f": "élément 3 (les 5 features les plus corrélées) : ces colonnes sont-elles modifiées ?", "379a5ad78da759b89d7e5305c8ccb60e56c9e38281a449670665bfb687ddfaee": "élément 1 (la colonne constante) : les colonnes qui restent sont-elles modifiées ?", "6ac8359e148b163c89c71b2d3ffa678829b3e769655bde5c983336f87afa764b": "élément 2 (la PCA) : ses composantes sont-elles des colonnes d'origine ?", "ee5d2d27f3631d3b1f8fcea2513131053dee802deefd4d7c73f0fc13dbf7996a": "élément 5 (VarianceThreshold) : que fait-il des colonnes qu'il garde ?"}}
📝 Ex 12.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "42fd8f4982d32fd0c516fe6800825a109641d2a398a32e8e4757

**12.Q9 — Ce que fait (et ne fait pas) une PCA**

In [11]:
wb.record("12.Q9a", True, mistakes={"relis la définition de la première composante (fiche §12.7.1)": False})
wb.record("12.Q9b", True, mistakes={"chaque composante est cherchée parmi les directions orthogonales aux précédentes": False})
wb.record("12.Q9c", False, mistakes={"la PCA ne reçoit que X, jamais y : elle est non supervisée": True})
wb.record("12.Q9d", True, mistakes={"une composante est une somme pondérée des features centrées, rien d'autre": False})
wb.record("12.Q9e", True, mistakes={"en grammes, les nombres sont 1 000 fois plus grands : que devient la variance de cette feature ?": False})
wb.record("12.Q9f", False, mistakes={"la PCA ne regarde pas les labels : une direction de petite variance peut séparer les classes": True})

WB_ANSWER {"hash": "d48b7df36fb19ee0561b2bd5c8b3447f76d4ef610b2adb8b93c7750cf6984dd7", "id": "12.Q9a", "kind": "bool", "mistakes": {"96e8d8e850e9a2165ac3a96a90ff6706d39dd72bd2d758de979e7657f5461d5f": "relis la définition de la première composante (fiche §12.7.1)"}}
📝 Ex 12.Q9a : réponse enregistrée (bool).
WB_ANSWER {"hash": "b23ebf4f942632b8b51c945bc536c0ea2b4a70849e58ad5528c165f417a3acaf", "id": "12.Q9b", "kind": "bool", "mistakes": {"f5f174f3f633b0ffcff1591c6ce54a0a89728bb6e3268867a0af7d950cddc729": "chaque composante est cherchée parmi les directions orthogonales aux précédentes"}}
📝 Ex 12.Q9b : réponse enregistrée (bool).
WB_ANSWER {"hash": "fb9c6481d07a8a524765a9257f0fe377025732a09862758b7f09ec1526187671", "id": "12.Q9c", "kind": "bool", "mistakes": {"fc7c5f0ed41e501e34a97cf8fd097dff405e94250ef3cdf657d89ce4c84c9d66": "la PCA ne reçoit que X, jamais y : elle est non supervisée"}}
📝 Ex 12.Q9c : réponse enregistrée (bool).
WB_ANSWER {"hash": "f9567f930952aef608020ede6108dfc5b8fc784d

**12.Q10 — Quelle découpe pour ces données ?**

In [12]:
wb.record("12.Q10a", "ELCELC", mistakes={"élément 2 (l'électrocardiogramme) : son propre maximum, c'est le maximum de quelle tranche du tableau ?": "ECCELC",
          "élément 5 (Normalizer) : la norme de quoi, d'une ligne ou d'une colonne ?": "ELCECC",
          "élément 5 (Normalizer) : la norme utilise-t-elle d'autres cases que celle qu'on transforme ?": "ELCEEC",
          "élément 1 (diviser par 255) : 255 dépend-il des autres pixels de l'image ?": "LLCELC",
          "élément 1 (diviser par 255) : 255 est-il calculé sur les autres images ?": "CLCELC",
          "élément 4 (un taux fixé) : ce taux dépend-il des autres prix de l'exemple ?": "ELCLLC",
          "élément 4 (un taux fixé) : ce taux est-il calculé sur les autres exemples ?": "ELCCLC",
          "élément 6 (la médiane de chaque colonne) : sur quelles valeurs est-elle calculée ?": "ELCELE"})
wb.record("12.Q10b", True, mistakes={"une telle transformation n'utilise aucune autre ligne : par où l'information circulerait-elle ?": False})

WB_ANSWER {"hash": "ba9e380f4ccf16c7abfb53fc3cca40a41c7c298fbee2993feb6e06b3589a72c9", "id": "12.Q10a", "kind": "str", "mistakes": {"0310174330988b9541af5fac3815388a1d2f51d72e799fc318875866a6c02f3a": "élément 1 (diviser par 255) : 255 est-il calculé sur les autres images ?", "2a65f70afd633e19018f22cd2795558bab126adb37a5946a041a9e15ddbc36dd": "élément 5 (Normalizer) : la norme utilise-t-elle d'autres cases que celle qu'on transforme ?", "331dd1a9de469e5786809f50dee87a2ededc0166bf7e60d407e6946b546a4493": "élément 2 (l'électrocardiogramme) : son propre maximum, c'est le maximum de quelle tranche du tableau ?", "45b2c2cc2ec3bdfe340cc4a67e9e6ec0e0cfa5378a4678ff2376cd55c148b43f": "élément 4 (un taux fixé) : ce taux est-il calculé sur les autres exemples ?", "595ec04c2f2d7231b285cf7b13e826b2432896141b2edda51ae688df1467170b": "élément 1 (diviser par 255) : 255 dépend-il des autres pixels de l'image ?", "5b61e2a45481d700d4faa798fbdf9cabe16c03b1c517c2d318ba691f213d5949": "élément 5 (Normalizer) 

**12.Q11 — Où se cache la fuite ?**

In [13]:
wb.record("12.Q11a", "ACE", mistakes={"F : diviser par une constante fixée d'avance n'apprend rien des données": "ACEF",
          "une procédure qui regarde la cible sur tout le dataset manque": "AC",
          "D : le scaler n'a vu que l'entraînement": "ACDE",
          "une imputation calculée avant le découpage apprend aussi quelque chose du test": "AE",
          "B : le pipeline est réajusté en entier dans chaque fold": "ABCE",
          "une standardisation faite avant la validation croisée a vu tous les folds": "CE"})
wb.record("12.Q11b", False, mistakes={"cross_val_score reçoit des données déjà transformées : peut-il défaire ce fit ?": True})

WB_ANSWER {"hash": "e655d9cb0e303d46299aa5ed7a233397cabef53baa51e288dce7933af37c8384", "id": "12.Q11a", "kind": "str", "mistakes": {"45d332ac38d2a1061bea3c4966ba7ec2cfc78a34b566c7d153feed0d6e3f80f9": "D : le scaler n'a vu que l'entraînement", "53b9132be7910768fdb60c6960bc4ce582f7b64251633c36c0758e36047a3146": "une procédure qui regarde la cible sur tout le dataset manque", "6cc6c15d46188a21938bcad144ef2c4a601392c99ea81b0da0a1fc4a20a315f6": "B : le pipeline est réajusté en entier dans chaque fold", "a78f1486abe88bd6ef268f485c7ac6109a42cce6e6bb95bae5e811be53e0acec": "une imputation calculée avant le découpage apprend aussi quelque chose du test", "cb991022ba0536509b6f92188357e66f4ad7f0d7eedd4ce5a567dcafeb059ea1": "une standardisation faite avant la validation croisée a vu tous les folds", "ee0996fe3523193d8d131de844e7422333b21da87d62940bc7d8ce40e3baa14c": "F : diviser par une constante fixée d'avance n'apprend rien des données"}}
📝 Ex 12.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash

**12.R1 — Représentation, évaluation, optimisation — où placer la préparation ?**

In [14]:
wb.record("12.R1a", "A", mistakes={"ajouter x₁² change l'ensemble des fonctions que le modèle peut exprimer": "C",
          "la mesure de la qualité d'une solution ne change pas": "B"})
wb.record("12.R1b", "C", mistakes={"sans pénalité, une régression sur des features standardisées exprime exactement les mêmes fonctions": "A",
          "la loss mesure toujours l'écart aux cibles de la même façon": "B"})
wb.record("12.R1c", "B", mistakes={"le F1-score juge les solutions, il n'en cherche pas": "C",
          "choisir la mesure ne change pas les modèles possibles": "A"})
wb.record("12.R1d", True, mistakes={"une information absente des features ne peut pas être exprimée, quel que soit l'optimiseur": False})

WB_ANSWER {"hash": "213ac3e081ce31afd168f481457c13a931cda6a386c29140872182d4c73244f8", "id": "12.R1a", "kind": "str", "mistakes": {"2d49afde6eedbf4866ef05fbaa65ae799bcb4c12a238b4d9ca853fed4d9858bd": "ajouter x₁² change l'ensemble des fonctions que le modèle peut exprimer", "b96b6b0fc55bec5d3d7abcefd25a9af29b4a4f0c400897fe14d0d94581369520": "la mesure de la qualité d'une solution ne change pas"}}
📝 Ex 12.R1a : réponse enregistrée (str).
WB_ANSWER {"hash": "34c7d8d7a3221403bfbbb4b6642baef3c2d7635a3da7552b8bbe3d7bafae62d7", "id": "12.R1b", "kind": "str", "mistakes": {"90aec21c7ffd89c3794156987eab2bdad278ac16b7072a52e70ad9a391f78022": "sans pénalité, une régression sur des features standardisées exprime exactement les mêmes fonctions", "e7535734141f34aee0465f01a6ab6a1b3715c33b06f59f834a31bdc60a60a906": "la loss mesure toujours l'écart aux cibles de la même façon"}}
📝 Ex 12.R1b : réponse enregistrée (str).
WB_ANSWER {"hash": "4e96011da97756967608dcdf54487f1633b45eeda31785c104359bc6135e0d28"

**12.R2 — Pourquoi la pénalité ridge dépend de l'échelle des features**

In [15]:
wb.record("12.R2a", "C", mistakes={"x' = 100 x : pour garder w' x' = w x, le poids doit compenser le facteur 100": "B",
          "sans changer le poids, la contribution de la feature serait multipliée par 100": "A",
          "w' x' doit valoir w x : de quel facteur faut-il diviser w ?": "D"})
wb.record("12.R2b", 100 ** 2, mistakes={"relis la pénalité : α w'², c'est α fois quoi ?": 100},
          fractional="on demande par combien la pénalité est divisée : un nombre plus grand que 1")
wb.record("12.R2c", "B", mistakes={"en centimètres, le poids devient plus petit : et sa pénalité ?": "A",
          "la pénalité dépend de la taille du poids, qui dépend de l'unité": "C"})
wb.record("12.R2d", True, mistakes={"x' = 100 x a les mêmes z-scores que x : la standardisation efface l'unité": False})

WB_ANSWER {"hash": "75c11c1eda76543179496451443a2fa90eee1ff66cf2e695642712ae2d7bad80", "id": "12.R2a", "kind": "str", "mistakes": {"1be13aea69ba015cdd2981ef35362b13e7469f71c6aef5c92b5c5831637eed2a": "x' = 100 x : pour garder w' x' = w x, le poids doit compenser le facteur 100", "32e1d9f696d30a855107d8adddcc3de2e00ebf7329814f5892e286457aeb9715": "w' x' doit valoir w x : de quel facteur faut-il diviser w ?", "c528344c31ab909e3e49d405644283b9b499ee78d25a419f916eba8b7adda0be": "sans changer le poids, la contribution de la feature serait multipliée par 100"}}
📝 Ex 12.R2a : réponse enregistrée (str).
WB_ANSWER {"fractional": "on demande par combien la pénalité est divisée : un nombre plus grand que 1", "hash": "d9ae465b184315e798ac9b706770ec069419541ca2409fd5e9fb98508e036cf7", "id": "12.R2b", "kind": "int", "magnitude": 4, "mistakes": {"de4fc7e082f9e0df3e055f69a7376df25734dc6295582aaaae44e0049d5adca3": "relis la pénalité : α w'², c'est α fois quoi ?"}, "sign": 1}
📝 Ex 12.R2b : réponse enregi

**12.R3 — Descente de gradient dans une vallée très allongée**

In [16]:
wb.record("12.R3a", GRAD_R3, mistakes={"la dérivée de 100 y² par rapport à y n'est pas 100": [2, 100],
          "la dérivée de x² par rapport à x n'est pas 1": [1, 200]})
wb.record("12.R3b", 0.01, decimals=4, mistakes={"ce pas annule le facteur : c'est le plus rapide sur y, pas la limite de divergence": 0.005,
          "la direction x n'est pas la plus contraignante : regarde le facteur de y": 1.0,
          "calcule le facteur 1 − 200η pour ce pas : il sort de [−1, 1]": 0.02})
wb.record("12.R3c", 1 - 2 * 0.009, decimals=3, mistakes={"la dérivée de x² n'est pas x : refais le pas sur x": 0.991,
          "−0,8 est le facteur de y ; la question porte sur x": -0.8})
wb.record("12.R3d", 1.0, decimals=2, mistakes={"après le changement d'échelle, recalcule la dérivée de y'²": 0.01,
          "c'est le pas qui annule le facteur, pas la limite de divergence": 0.5,
          "écris 100 y² en fonction de y' = 10 y, puis refais le calcul de b) avec ce nouveau terme": 0.1})
wb.record("12.R3e", True, mistakes={"quand la vallée redevient ronde, le même learning rate convient aux deux directions": False})

WB_ANSWER {"decimals": 0, "element_hashes": ["c8202953f9957f9e04a1ce73de26a6342d2b281eaa4deff24eb7052f1f8e809f", "d815704a26573cdf57cac4417c54d63098c7dc0bf4519ebc1a8615d9c762d65e"], "hash": "674762ebb0907d945869a096201fca5f6dd8cbe59384e83da8278dbc28b495d0", "id": "12.R3a", "integer": true, "kind": "array", "mistakes": {"6c6cf904d86812c1d361ba563a4fa6d955dfda1d4ed425591351503a7eb4189f": "la dérivée de 100 y² par rapport à y n'est pas 100", "759d4e79b64c5940befac1b757544e3e05bbf5dbc81e689fbdc1a032f5165037": "la dérivée de x² par rapport à x n'est pas 1"}, "shape": [2]}
📝 Ex 12.R3a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "dac144d251698415b9dbc5d2bb94395fb3db18b6e60c930418f3fdad779b2582", "hash_coarse": "82ca1a93b42d783939b8454438a7d992a22e225aca506addcd687086d7ab9daa", "id": "12.R3b", "kind": "float", "magnitude": -2, "mistakes": {"a582ff862143c0266ca65e4da57a03fd25863ea02cf9653a9f5dd640b6d75ccf": "la direction x n'est pas la plus contraignante : re

**Ex 12.1 — One-hot à la main sur Penguins**

In [17]:
wb.record("12.1a", 3 + 2 + 1, mistakes={"n'oublie pas flipper_length_mm, gardée telle quelle": 5,
          "compte une colonne par catégorie dans chaque bloc, plus la feature numérique": 3})
wb.record("12.1b", [0, 0, 1, 1, 0, 185], mistakes={"bloc sex : dans l'ordre alphabétique, female vient avant male": [0, 0, 1, 0, 1, 185],
          "bloc island : Torgersen est-elle la première île dans l'ordre alphabétique ?": [1, 0, 0, 1, 0, 185]})
wb.record("12.1c", 2 + 1 + 1, mistakes={"drop=\"first\" retire une colonne par feature catégorielle, rien à la feature numérique": 3,
          "deux features catégorielles : une colonne retirée pour chacune": 5})
wb.record("12.1d", [0, 0, 1, 220], mistakes={"drop=\"first\" retire la colonne de la PREMIÈRE catégorie de chaque bloc, dans l'ordre alphabétique": [1, 0, 0, 220],
          "drop=\"first\" retire une colonne dans chaque bloc catégoriel : compte les colonnes qui restent": [1, 0, 0, 0, 1, 220]})
wb.record("12.1e", 3 + 1 + 1, mistakes={"if_binary ne retire une colonne qu'aux features à deux catégories": 4,
          "une feature à deux catégories perd une colonne avec if_binary": 6})
wb.record("12.1f", [0, 0, 0, 1, 0, 195], mistakes={"une île inconnue n'est rangée dans aucune des catégories apprises": [0, 0, 1, 1, 0, 195],
          "une île inconnue ne prend pas la place de la première catégorie apprise": [1, 0, 0, 1, 0, 195]})
wb.record("12.1g", 2, mistakes={"les codes commencent à 0": 3,
          "l'ordre est alphabétique : où se range Torgersen parmi les trois îles ?": 1})

WB_ANSWER {"hash": "236187d655b48625bcc541624a99982d8d5713267578dc6c716253edb89b3192", "id": "12.1a", "kind": "int", "magnitude": 0, "mistakes": {"23e83f369666429e246547ce94b1e4ae3e2bbb819a267bdb97973ca5a6e0b356": "n'oublie pas flipper_length_mm, gardée telle quelle", "cb0c0c4ac85b19fb97404fadf24af02eed68d8e8da60f4fcfb80e71ce6b0f937": "compte une colonne par catégorie dans chaque bloc, plus la feature numérique"}, "sign": 1}
📝 Ex 12.1a : réponse enregistrée (int).
WB_ANSWER {"decimals": 0, "element_hashes": ["bb72c6450b5bc2ed3344c65bf980b27aa7595a73d837d69dae76bb2c06831bb4", "ca0324ad09a5e1bd15843490f88894fc6fd0ca4bfdf36691125f6fa184072eee", "47fec770907f7c838c510bf74ff36a0c1a3906246fb722aaf849d6a46ba1fccb", "d3f2c9c666011a287f3f75390586bb72a6b7ec60b15d7ba4bf37a9d4aadd036c", "b2b093ca6f19f968ccd1df62671909444f3c509d15c55ce0495f8efe2f3a2e51", "ef9bf01031a0ffad5fbee3e646d2611a58879f0dd88281d0e47f19575c8f20f4"], "hash": "0ba8162ff226b35f68f10559635a255fc88e49534a29ebacaad2512469bca7c3", "

**Ex 12.2 — Min-max et z-score de cinq valeurs**

In [18]:
wb.record("12.2a", MU_122, decimals=2, mistakes={"la moyenne n'est pas la valeur du milieu (la médiane)": 21})
wb.record("12.2b", SD0_122, decimals=3, mistakes={"tu as divisé la somme des carrés par n − 1 (ddof = 1) ; l'énoncé demande ddof = 0, comme StandardScaler": round(SD1_122, 3),
          "ton écart-type est calculé avec n − 1 au dénominateur (ddof = 1) ; ici, ddof = 0": round(SD1_122, 2),
          "tu as donné une variance : l'écart-type est sa racine": 16,
          "tu as donné une variance, et avec n − 1 : l'écart-type demandé est la racine de la variance avec n": 20})
wb.record("12.2c", float(minmax(21, 14, 25)), decimals=3, mistakes={"retire le minimum (pas la moyenne) avant de diviser par l'étendue": 0.091,
          "divise par l'étendue max − min, pas par le maximum": 0.28,
          "tu as divisé la valeur par le maximum : retire d'abord le minimum": 0.84})
wb.record("12.2d", (17 - MU_122) / SD0_122, decimals=2, mistakes={"17 est sous la moyenne : son z-score est négatif": 0.75,
          "divise l'écart à la moyenne par l'écart-type": -3,
          "ton écart-type est calculé avec n − 1 au dénominateur ; l'énoncé demande ddof = 0": -0.67,
          "tu as divisé l'écart à la moyenne par la variance : divise par l'écart-type": -0.19})
wb.record("12.2e", (25 - MU_122) / SD0_122, decimals=2, mistakes={"divise l'écart à la moyenne par l'écart-type": 5,
          "ton écart-type n'est pas celui de l'énoncé : vérifie le dénominateur de la variance (ddof = 0)": 1.12,
          "tu as divisé l'écart à la moyenne par la variance : divise par l'écart-type": 0.31})
wb.record("12.2f", (25 - MU_122) / SD1_122, decimals=3, mistakes={"quel dénominateur as-tu pris pour la variance ? Series.std() divise par n − 1": 1.25})
wb.record("12.2g", float(minmax(21, 14, 25, -1, 1)), decimals=3, mistakes={"vérifie la formule vers [a, b] : as-tu bien pris a = −1 et b = 1 ?": 0.636,
          "il manque la multiplication par b − a = 2": -0.364,
          "il manque un terme : après la multiplication par b − a, la formule ajoute a": 1.273})

WB_ANSWER {"decimals": 2, "hash": "214de49abaf512922a87dbd3378e3c42a4d5bd70efb732a586f0beee2a57e5e3", "hash_coarse": "105ed40392ac538965f861836a0cda0c4563b5632e63dab11fb454dcb83ff989", "id": "12.2a", "kind": "float", "magnitude": 1, "mistakes": {"a976010f5910ffb38adcb24c4bd22a7fd749bfef92ff084b4adcbebc6137f4a0": "la moyenne n'est pas la valeur du milieu (la médiane)"}, "sign": 1}
📝 Ex 12.2a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "c7bbbf947ae2bebc9f236293c9a9e9ab74fa84c34f0b023eaa91b22af18799b0", "hash_coarse": "ca9c431bbb38dea1cab48881b92efcc511ff321d78202e938192bb2b6f510495", "id": "12.2b", "kind": "float", "magnitude": 0, "mistakes": {"68136f7013b13248e46df26492ec3be7d124f2092e0e3ebbec9047b846813b52": "ton écart-type est calculé avec n − 1 au dénominateur (ddof = 1) ; ici, ddof = 0", "be69fbf60b7df875d000440737f0c416ea58a68fb3554a7afb324589353eeeb8": "tu as donné une variance : l'écart-type est sa racine", "f4613ff3062779c005b710a3f0202eb8e059

**Ex 12.3 — Mise à l'échelle univariée ou multivariée**

In [19]:
wb.record("12.3a", minmax(X_123, LO_123, HI_123).tolist(), decimals=2, mistakes={"retire d'abord le minimum de chaque feature": [0.67, 0.8, 0.0]})
wb.record("12.3b", minmax(X_123, GLO_123, GHI_123).tolist(), decimals=2, mistakes={"en multivarié, le minimum et le maximum sont communs aux trois features : vérifie lesquels tu as pris": [0.5, 0.8, 0.5],
          "retire le minimum global, qui est négatif, avant de diviser": [0.4, 0.08, 0.0],
          "le minimum global vaut −20 : le retirer, c'est ajouter 20": [0.0, -0.32, -0.4],
          "le minimum et le maximum globaux viennent des trois features, pas d'une seule": [1.0, 0.6, 0.5]})
wb.record("12.3c", (HI_123[1] - LO_123[1]) / (GHI_123 - GLO_123), decimals=3, mistakes={"en multivarié, f2 ne remplit plus tout [0, 1] : elle subit la même dilatation que les autres features": 1.0,
          "l'étendue globale va du plus petit minimum au plus grand maximum des trois features": 0.125,
          "l'étendue globale va du plus petit minimum au plus grand maximum des trois : pas seulement ceux de f3": 0.13})
wb.record("12.3d", True, mistakes={"le min-max multivarié applique la même dilatation à toutes les features": False})
wb.record("12.3e", "B", mistakes={"un min-max par colonne donne à chaque magasin son propre minimum : une même valeur transformée ne veut plus dire le même montant": "A"})

WB_ANSWER {"decimals": 2, "element_coarse": [["f1abd9f96b2d8507e79e3025884b4d6d671fb3651e1e202d614cb329df90f37b"], ["805718ae3b5dfaa188b17ec3e0678e2a299b315c79edfab360e2694344276a70"], ["a4a1e2bc73e2d878cce9e103aeaf8f6e92c8469eac8edda6927535cf70dfff4e"]], "element_hashes": ["9a3954ecaa3f90ac7ac1d60f64bc00482a35c5ba5fd2b09fbab75dc4218bd6fe", "812b0f907f9f42427e0061ee9e3b7573905d37ccdc7aac4e3069e7db3a63c679", "76dc67cabc24e8cdfa24913cdad8b993d82d6af88c3d4dddab7f13eb443a0a3f"], "hash": "bb9e287de01a0e6ac346bd5e7dce69eb84f7c098d8da4ed113d7f79d22e9111b", "id": "12.3a", "kind": "array", "mistakes": {"6c3cf1f9f9de4b06f2f52b79aa506ba148118502063620b1abaedeea80b7fa6f": "retire d'abord le minimum de chaque feature"}, "shape": [3]}
📝 Ex 12.3a : réponse enregistrée (array, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "element_coarse": [["0f31e2f30d85c1e1142f2adcef270148cfe6b2f2358f7fb6634c96b3d7a3f5a2"], ["86a2416e5300bf857b8696ada3a1c8b65b70de5b5a5c46f5c0ad1a4fdcdf9c38"], ["057b0307e717cc09779dd2921

**Ex 12.4 — Réappliquer la transformation : −10 °C, −50 °C et retour aux voitures**

In [20]:
wb.record("12.4a", float(minmax(-10, T_LO, T_HI)), decimals=3, mistakes={"le numérateur retire le minimum, pas le maximum": -0.733,
          "le minimum vaut −18 : le retirer, c'est ajouter 18": -0.933,
          "retire d'abord le minimum, puis divise par l'étendue": -0.333,
          "l'étendue vaut max − min = 12 − (−18) : attention au signe": 1.333,
          "l'étendue max − min est positive : 12 − (−18), soit 12 + 18": -1.333})
wb.record("12.4b", C_LO + 0.40 * (C_HI - C_LO), decimals=0, mistakes={"l'inverse ajoute aussi le minimum des comptages": 300,
          "l'inverse multiplie par l'étendue des comptages, max − min, pas par le maximum": 468,
          "deux oublis : l'inverse multiplie par l'étendue (max − min), puis ajoute le minimum": 348,
          "−6 voitures ? Vérifie le minimum et le maximum que tu as utilisés pour revenir en arrière": -6})
wb.record("12.4c", float(minmax(-50, T_LO, T_HI)), decimals=3, mistakes={"retire d'abord le minimum, puis divise par l'étendue": -1.667,
          "attention au signe : cette nuit est bien plus froide que le minimum d'entraînement": 1.067,
          "le minimum vaut −18 : le retirer, c'est ajouter 18": -2.267,
          "sans clip=True, transform ne borne rien : applique la formule telle quelle": 0})
wb.record("12.4d", float(minmax(20, T_LO, T_HI)), decimals=3, mistakes={"retire d'abord le minimum, puis divise par l'étendue": 0.667,
          "le minimum vaut −18 : le retirer, c'est ajouter 18": 0.067,
          "sans clip=True, transform ne borne rien : applique la formule telle quelle": 1.0})
wb.record("12.4e", False, mistakes={"une valeur hors de la plage d'entraînement sort de [0, 1] : relis la fiche §12.5.3": True})
wb.record("12.4f", C_LO + (-0.1) * (C_HI - C_LO), decimals=0, mistakes={"l'inverse s'applique aussi hors de [0, 1] : n'oublie pas d'ajouter le minimum": -75,
          "rien ne borne une prédiction hors de [0, 1] : l'inverse s'applique telle quelle": 120})
wb.record("12.4g", (-10 - (-3)) / 6, decimals=3, mistakes={"−10 est sous la moyenne : z est négatif": 1.167,
          "retire la moyenne avant de diviser par l'écart-type": -1.667,
          "la moyenne vaut −3 : la retirer, c'est ajouter 3": -2.167})
wb.record("12.4h", 480 + 0.5 * 150, decimals=0, mistakes={"l'inverse multiplie par l'écart-type avant d'ajouter la moyenne": 480.5,
          "n'oublie pas d'ajouter la moyenne": 75})
wb.record("12.4i", False, mistakes={"chaque grandeur a son transformateur : la cible se ramène avec le sien": True})

WB_ANSWER {"decimals": 3, "hash": "3196a78abcbf2745194696f2ceec0e49f908475781504b54bafe499b34001f25", "hash_coarse": "9a5a070c4b8070098c74a490881caddbe75a33d33e7711edce028f2075d91bf8", "id": "12.4a", "kind": "float", "magnitude": -1, "mistakes": {"0808f7fed4ad5de4862d0dc17da344836c94c65e397a4e25802f5440f8e6c3f5": "le minimum vaut −18 : le retirer, c'est ajouter 18", "76d0b7a0aa61e6fc8333494a213314ff5608583f5bede21b077d1a7028d47342": "le numérateur retire le minimum, pas le maximum", "e92b852546b494c944e4fac5a272efafcf3152a20b5e02cc1ee6ad739d65750a": "l'étendue max − min est positive : 12 − (−18), soit 12 + 18", "f745c8a247fcc342dcdc4ef31d48ec3fd34fe54d2a95ffce56a1da0de6995e3c": "l'étendue vaut max − min = 12 − (−18) : attention au signe", "f88e51f71eeb612e5b9edbe57fb11118284c6f3f61ad6d0bd584ea420049191f": "retire d'abord le minimum, puis divise par l'étendue"}, "sign": 1}
📝 Ex 12.4a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 0, "hash": "15ed3ab093380cf03dc92fd

**Ex 12.5 — Trois découpes d'un même tableau : échantillon, feature, élément**

In [21]:
wb.record("12.5a", minmax(X_125[1], COL_LO, COL_HI).tolist(), decimals=2, mistakes={"par feature : chaque colonne avec son propre minimum et son propre maximum, pas chaque ligne": [0.0, 0.29, 1.0]})
wb.record("12.5b", row_minmax(X_125[0]).tolist(), decimals=3, mistakes={"par échantillon : la ligne avec son propre minimum et son propre maximum, pas chaque colonne": [0.0, 0.0, 0.0]})
wb.record("12.5c", float(X_125[1, 0] / np.linalg.norm(X_125[1])), decimals=3, mistakes={"divise par la norme (la racine de la somme des carrés), pas par la somme": 0.097,
          "la norme se calcule sur la ligne, pas sur la colonne": 0.507,
          "la norme est la racine de la somme des carrés : as-tu pris la racine ?": 0.006})
wb.record("12.5d", (0.9 * X_125[2]).tolist(), decimals=2, mistakes={"à 0,9 € pour 1 $, on multiplie, on ne divise pas": [5.56, 6.67, 33.33]})
wb.record("12.5e", "A", mistakes={"une conversion à un taux fixé d'avance n'apprend rien des données": "AD",
          "chaque ligne se transforme avec son propre minimum et son propre maximum : rien n'est appris des autres lignes": "AB",
          "la norme d'une ligne ne dépend que de cette ligne": "AC",
          "B et C sont par échantillon : utilisent-elles les autres lignes de l'entraînement ?": "ABC",
          "toutes ne s'apprennent pas : lesquelles utilisent les autres exemples de l'entraînement ?": "ABCD"})
wb.record("12.5f", minmax(NEW_125, COL_LO, COL_HI).tolist(), decimals=2, mistakes={"par feature, chaque valeur se transforme avec le minimum et le maximum de sa colonne d'entraînement : vérifie lesquels tu as pris": [0.0, 0.0, 1.0],
          "rien ne borne une donnée nouvelle : pas de clip": [0.25, 0.0, 1.0]})
wb.record("12.5g", row_minmax(NEW_125).tolist(), decimals=2, mistakes={"par échantillon, la ligne se transforme avec son propre minimum et son propre maximum : vérifie lesquels tu as pris": [0.25, -0.5, 1.5]})

WB_ANSWER {"decimals": 2, "element_coarse": [["ea253eb2cdc3ea459312cff44fa68f2cee76fe6ab79ecd37dc0e5a439b7f51ca"], ["f9144b08732946e1ed65533417c8a05e156d6b213cd1dafe4bc6eefbca1caa8e"], ["bc6fb20c165897700b7089666bd35f082b9ef7ccd9e3d5baf824b078945601d1"]], "element_hashes": ["94d7c0494889215b28bbb7c256c2e5d520c5adf1d05ce2e3391c1ccd16a1d330", "138ddb4113fd8e524c02086d48001772d1825e60c4032a53ac3648436cffd78b", "2bceba6cdabe79726eff650931405080260cb333bd3a78efac9bb99b03354bda"], "hash": "40807834cef350ea3c0c356ba60d8482fa1267de93da529c7380b29ba2132fe9", "id": "12.5a", "kind": "array", "mistakes": {"615159a6189c84de4070ec71909c80d55b00f3f7e7ad2e53dba5dd286bf4590f": "par feature : chaque colonne avec son propre minimum et son propre maximum, pas chaque ligne"}, "shape": [3]}
📝 Ex 12.5a : réponse enregistrée (array, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "element_coarse": [["49da806d9f79b3f12ac36dec20661ba65a24fa5a290ea5108c08c049eed2dc9d"], ["bdaea395b99f043ed0bb9e533512894ac1a2d79aba8158

**Ex 12.7 — PCA à la main en 2D : covariance, axe principal, projection**

In [22]:
wb.record("12.7a", MU_127.tolist(), decimals=2)
wb.record("12.7b", COV_127.tolist(), decimals=2, mistakes={"divise par n − 1 (ddof = 1), comme np.cov": [[4.0, 1.6], [1.6, 1.6]],
          "tu as la somme des produits des écarts : divise par n − 1": [[20, 8], [8, 8]],
          "centre d'abord les points : la covariance porte sur les écarts au point moyen": [[25, 17], [17, 13.25]]})
wb.record("12.7c", (COV_127 @ np.array([2.0, 1.0])).tolist(), decimals=2, mistakes={"l'énoncé demande le produit par (2, 1), sans normaliser le vecteur": [5.37, 2.68]})
wb.record("12.7d", LAMBDA_127, decimals=2, mistakes={"12 est une coordonnée du produit, pas la valeur propre : relis la définition Σ u = λ u": 12})
wb.record("12.7e", TOTAL_127, decimals=2, mistakes={"la variance totale ne compte pas les covariances": 11})
wb.record("12.7f", LAMBDA_127 / TOTAL_127, decimals=3, mistakes={"tu as pris la variance de la première feature ; la composante a la sienne (question d)": 0.714,
          "la variance totale ne compte pas les covariances : revois e)": 0.545})
wb.record("12.7g", T_127.tolist(), decimals=3, mistakes={"centre d'abord les points (retire le point moyen)": [1.342, 4.025, 4.919, 6.708, 7.603],
          "u est unitaire : divise (2, 1) par sa norme": [-8, -2, 0, 4, 6]})
wb.record("12.7h", float(np.var(T_127, ddof=1)), decimals=2, mistakes={"ddof = 1 : divise la somme des carrés par n − 1": 4.8})
wb.record("12.7i", REC_D.tolist(), decimals=2, mistakes={"ajoute le point moyen à t u": [1.6, 0.8],
          "la reconstruction reste sur la droite principale : elle ne retrouve pas le point": [5, 5],
          "u est unitaire : la reconstruction ajoute t u, pas t (2, 1)": [7.58, 4.79]})
wb.record("12.7j", float(np.linalg.norm(D_127 - REC_D)), decimals=3, mistakes={"tu as la distance au point moyen, pas à la reconstruction": 2.236,
          "tu as la distance au carré : prends sa racine": 1.8})
wb.record("12.7k", T_D / math.sqrt(LAMBDA_127), decimals=3, mistakes={"divise par la racine de la valeur propre, pas par la valeur propre": 0.298,
          "le whitening divise encore la coordonnée par l'écart-type de sa composante": 1.789})
wb.record("12.7l", float(np.var(XC_127 @ V_127, ddof=1)), decimals=2, mistakes={"ddof = 1 : divise la somme des carrés par n − 1": 0.8,
          "v est unitaire : divise (−1, 2) par sa norme avant de projeter": 5,
          "les points ne sont pas tous sur l'axe principal : calcule leurs coordonnées sur v": 0})
wb.record("12.7m", False, mistakes={"multiplier une feature par 10 multiplie sa variance par 100 : vers où l'axe principal tourne-t-il ?": True})

WB_ANSWER {"decimals": 2, "element_coarse": [["59816095a5ffcf91e4db0bacf67201c05373db8ed176602981ac2dfb8e9b50cd"], ["3b2003bf45891ee8ec26dd9bc3c471e8e0dea3a3c5fef1ebee52e6df4fd2ac27"]], "element_hashes": ["6119a61b89165a75f2f27c26fb89bd6fab34339ae8e1a5045fd043674ad36bc3", "5f6119baf6c0255f2afd3dfd1c2bedb4abf36fd3e168477572a5ff417b113e20"], "hash": "c1a7d273b723feffc3d53e925fd9b2348c373227583343e27bb00869b4e19baf", "id": "12.7a", "kind": "array", "shape": [2]}
📝 Ex 12.7a : réponse enregistrée (array, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "element_coarse": [["ad5395b615044b61e77970caa245fc4b3f7b5d96c2260012924d357c2a4669ec"], ["dcc0b1d5fcb223aa6c60eea7a72854622e20eabd7a976b37fa565cfca771b9de"], ["f8aef68db9f7addda47319cad01db39197ae76632e7570ad927d77b10eac03c9"], ["83beaa0bbc35c968537288de90a02a63157855579ec8e5233ac017784b2b9aed"]], "element_hashes": ["3d64067b13276a7085a2172691fa274614569afd1c5f48714c8906a058233f1a", "72565efecece47de9f27ba8296f30729b6227b10fe7002ce51e0079002dce2f2"

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire, pour chaque étape d'une préparation, ce qu'elle apprend des données, et donc où faire son `fit` ?
2. Sais-tu choisir un encodage et une mise à l'échelle selon le type des données et le modèle qui suit ?
3. Sais-tu mener une PCA à la main en dimension 2, et dire ce qu'elle garde et ce qu'elle perd ?

**Pour aller plus loin** : le guide « Preprocessing data » et la page « Common pitfalls » de scikit-learn, et le tutoriel de J. Shlens sur la PCA, cités dans la fiche. La suite : le ch. 13 (classifieurs), où les k plus proches voisins et les SVM dépendront de tes mises à l'échelle, et pas les arbres de décision.

*Les exercices de code 12.11 à 12.33 (parties A à D, ta librairie `mylearn.preprocessing`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 12` pour obtenir le notebook complet.*